## In-Class Lecture 38-1 - Kalman Filter

In this lecture, you will write the two functions at the heart of a spacecraft sensor monitor: noise statistics for calibration, and a Kalman filter step with a measurement gate. The tests compare every answer to a tolerance, as you learned in Lecture 33. Then a mission driver uses your code to filter a temperature, catch a failing sensor, and write a file you can plot.

### Reviewing Opening your Environment on the Notre Dame Machines

<b>1.</b> Log into the Notre Dame Machines as you performed on Homework 1 using Tabby Terminal 

Open Tabby Terminal on your machine.

Type in: <code>ssh -Y username@student13.cse.nd.edu</code>
<ul><li>"username" is your NetID.</li>
<ul>
<li>Example: my email is matt.morrison@nd.edu, but my NetID is mmorri22, so I would enter:<p>
<code>ssh -Y mmorri22@student13.cse.nd.edu</code><p>
The <code>-Y</code> is included to treat the Notre Dame machines as a trusted client<p>
</li>
</ul>
</ul>

2. Next, we will create a directory where you will save your In-Class Code. First, we will go into the Dropbox directory where you will submit your code:

<code>cd esc-courses</code>

<code>cd fa26-cse-20133.01</code>

<code>cd dropbox</code>

Make a directory for you to practice your code for this lecture

    mkdir inclass38
    cd inclass38

Enable script so you can run the most up-to-date versions of C and C++:

    scl enable gcc-toolset-15 bash

> <b>Note:</b> If you use the <code>-std=c2x</code> and get the following error, this means you did not run the enable command<br>
> <code>gcc: error: unrecognized command line option ‘-std=c2x’; did you mean ‘-std=c1x’?</code>

To obtain the code for today's lecture, perform the following commands:

    wget https://raw.githubusercontent.com/mmorri22/cse20133/main/inclass/lec38/setup.sh
    chmod a+rx setup.sh
    ./setup.sh
    
To confirm, perform the <code>ls</code> command. You will see the following:

    calibration.csv  kalman_filter.c  kalman_filter.h  Makefile  mission.csv  sensor_run.c  test_kf.c

### In-Class Reading Discussion

<font color = "red">Class Introduction Question #1 - Using the output of <code>noise_stats</code>, is this sensor accurate, precise, both, or neither? Explain using the bias and the standard deviation.</font>

<font color = "red">Class Introduction Question #2 - If the noise is Gaussian, about how often should a reading land more than 3 standard deviations from the expected value? Why does that make a 3-standard-deviation reading worth a second look?</font>

<font color = "red">Class Introduction Question #3 - Both filters cut the RMS error in half. What did they cost? Using the table around sample 36, explain the trade-off.</font>

<font color = "red">Class Introduction Question #4 - In <code>kalman_step</code>, what happens to the gain <code>K</code> when the sensor is very noisy? When the prediction is very uncertain? Why did <code>K</code> start near 0.5 and settle near 0.35?</font>

<font color = "red">Class Introduction Question #5 - Explain why <code>Q</code> = 0.001 never caught up after the step, and why <code>Q</code> = 1.0 had a larger RMS error than <code>Q</code> = 0.1.</font>

<font color = "red">Class Introduction Question #6 - What is the innovation? In <code>innovation.c</code>, why was the reading at sample 53 rejected but not treated as a sensor fault, while the readings after sample 45 in the second run were?</font>

### The Files

<ul>
    <li><code>kalman_filter.h</code> - the <code>kalman1d</code> and <code>kf_result</code> structures, and the two functions you will write. Read it first.</li>
    <li><code>kalman_filter.c</code> - your code.</li>
    <li><code>test_kf.c</code> - 16 tests. Every floating-point check uses a tolerance of 10<sup>-9</sup>, never <code>==</code>.</li>
    <li><code>sensor_run.c</code>, <code>calibration.csv</code>, <code>mission.csv</code> - the mission driver and the same data as the reading.</li>
</ul>

    make test

Four tests pass by coincidence. Make all 16 pass.

### Step 1 - <code>compute_stats</code> (Tests 1 to 5)

Compute the mean and the <b>sample</b> variance of <code>n</code> values, as in <code>noise_stats.c</code>: one pass for the mean, a second pass for the sum of squared differences, divided by <code>n - 1</code>. Return -1 if any pointer is <code>NULL</code> or <code>n</code> is less than 2. Why does a variance need at least two values?

### Step 2 - <code>kf_step</code>, the Normal Update (Tests 6 to 10)

Write one predict / gain / update cycle, as in <code>kalman_step</code> from Reading 38-2, and also fill in <code>*out</code>:
<ol>
    <li>Return -1 if <code>kf</code> or <code>out</code> is <code>NULL</code>.</li>
    <li><b>Predict:</b> <code>x_pred = kf->x</code> and <code>P_pred = kf->P + kf->Q</code>.</li>
    <li><b>Innovation:</b> <code>S = P_pred + kf->R</code>. Store <code>z - x_pred</code> in <code>out->innovation</code> and the innovation divided by <code>sqrt(S)</code> in <code>out->surprise</code>.</li>
    <li><b>Gain and update:</b> <code>K = P_pred / S</code>, then update <code>kf->x</code> and <code>kf->P</code>. Store <code>K</code> in <code>out->K</code> and 0 in <code>out->rejected</code>.</li>
</ol>

### Step 3 - The Gate (Tests 11 to 14)

Before the update, add the gate from <code>innovation.c</code>: if the absolute value of <code>out->surprise</code> is greater than <code>kf->gate</code>, reject the measurement.
<ul>
    <li>Set <code>out->rejected</code> to 1 and <code>out->K</code> to 0.0.</li>
    <li>Keep the prediction: <code>kf->x = x_pred</code>.</li>
    <li>Keep the <b>larger</b> uncertainty: <code>kf->P = P_pred</code>.</li>
</ul>

Test 13 checks that a rejected measurement still grows <code>P</code>. Why is that important? <b>Hint:</b> think about what happens to <code>S</code>, and to the gate, if the filter rejects several readings in a row.

### Step 4 - Convergence (Tests 15 and 16)

Test 15 runs your filter for 100 steps and checks that <code>K</code> settles to the exact steady-state gain from the theory, to 9 decimal places. If Steps 2 and 3 are right, it passes with no new code.

### Step 5 - Fly the Mission

    make mission

The driver calibrates <code>R</code> with your <code>compute_stats</code>, filters the mission with your <code>kf_step</code>, and writes <code>estimate.csv</code>. You should see the same two rejections as in Reading 38-2.

Use FileZilla to download <code>estimate.csv</code>, open it in a spreadsheet, and plot the true value, the measurement, and the estimate against the sample number. You have just reproduced the chart on Ama's slide 50 with your own filter.

### Step 6 - A Failing Sensor

    make bias

The sensor develops a +3.0 C bias at sample 45. Your filter should catch it at sample 47. <code>make</code> ends with <code>Error 1</code> because the driver returns <code>EXIT_FAILURE</code> for a confirmed fault.

### Step 7 - When the Filter Is Wrong

Now run the <b>healthy</b> sensor with a badly tuned filter:

    make mission Q=0.001

Nothing is wrong with the sensor, but the program declares a <b>sensor fault</b> at sample 37. Look at what was rejected, and plot the new <code>estimate.csv</code>.

With <code>Q</code> = 0.001, the filter is certain that the temperature can barely change. When the temperature really did jump at sample 36, the filter decided the <i>sensor</i> was lying. A spacecraft running this filter would switch away from a perfectly good sensor, and keep believing a temperature that was no longer true.

### Discussion

<ol>
    <li>In Step 7, the fault detector reported a fault that did not exist. What was actually wrong? Whose mistake was it: the sensor's, the code's, or the engineer who chose <code>Q</code>?</li>
    <li>How could the software tell the difference between "the sensor developed a bias" (Step 6) and "the temperature really changed" (Step 7)? <b>Hint:</b> recall the three redundant sensors from Lecture 37.</li>
    <li><code>Q</code> = 1.0 would never have caused the false alarm in Step 7. Why not use a large <code>Q</code> all the time?</li>
</ol>

## Compilation Run

    make test
    make mission
    make bias
    make analyze

When you are finished, all 16 tests print <code>PASS</code>, <code>make mission</code> rejects two readings and confirms no fault, <code>make bias</code> confirms a fault at sample 47, and <code>make analyze</code> reports no errors.